## Env and client creation

In [30]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helper Functions

In [45]:
# Helper functions
from anthropic.types import Message
def add_user_message(messages, message):
    user_message = {
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message
        }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None, tool_choice = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tool_choice:
        params["tool_choice"] = tool_choice
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join([ block.text for block in message.content if block.type == "text"])

## Tools

In [44]:
# Tools and Schemas

from datetime import datetime, timedelta

article_summary_schema = {
  "name": "article_summary",
  "description": "Generate a summary for an article given its title, author, and a list of key insights extracted from the content.",
  "input_schema": {
    "type": "object",
    "properties": {
      "title": {
        "type": "string",
        "description": "The title of the article"
      },
      "author": {
        "type": "string",
        "description": "The name of the article's author"
      },
      "key_insights": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "A list of key insights or takeaways from the article"
      }
    },
    "required": ["title", "author", "key_insights"]
  }
}

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [33]:
from datetime import datetime
from anthropic.types import ToolParam
def get_current_datetime(date_format = "%Y-%m-%d %H:%M:%S"):
    
    if not date_format:
        raise ValueError("date_format cannot be empty")
    
    return datetime.now().strftime(date_format)

get_current_datetime_schema =ToolParam({
    "name": "get_current_datetime",
    "description": "Get the current date and time, formatted according to the specified format string. Returns the current datetime as a formatted string.",
    "input_schema": {
        "type": "object",
        "properties": {
        "date_format": {
            "type": "string",
            "description": "A Python strftime format string for the output. Common examples: \"%Y-%m-%d %H:%M:%S\" (full datetime), \"%Y-%m-%d\" (date only), \"%H:%M:%S\" (time only). Defaults to \"%Y-%m-%d %H:%M:%S\" if not provided."
        }
        },
        "required": []
    }
})

## Tool Use -- single

In [34]:
messages = []

messages.append(
    {
        "role": "user",
        "content" : "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model = model,
    max_tokens= 1000,
    messages= messages,
    tools = [get_current_datetime_schema],
)
messages.append({
    "role":"assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ATkEQp4XojFzK7hWKtUGgv', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [35]:
result = get_current_datetime(**response.content[0].input)

In [36]:
messages.append(
    {
        "role": "user",
        "content" : [
            {
                "type"  : "tool_result",
                "tool_use_id" : response.content[0].id,
                "content": result,
                "is_error" : False
            }
        ]
    }
)

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ATkEQp4XojFzK7hWKtUGgv', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01ATkEQp4XojFzK7hWKtUGgv',
    'content': '09:05:36',
    'is_error': False}]}]

In [37]:
client.messages.create(
    model = model,
    max_tokens = 1000,
    messages=messages,
    tools = [get_current_datetime_schema]
)

Message(id='msg_01WECkEcsJgW1uKHedUtcfLY', container=None, content=[TextBlock(citations=None, text='The exact time is **09:05:36**.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=751, output_tokens=14, server_tool_use=None, service_tier='standard'))

## Tool Use -- Multi Turns

In [38]:
import json
def run_batch(invocations):
    batch_output = []
    
    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        tool_output = run_tool(name, args)
        
        batch_output.append({"tool_name" : name, "output" : tool_output})
        
    return batch_output


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_remainder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
        

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    
    tool_result_blocks = []
    for tool_request in tool_requests:
        
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_request.id,
                "content" : json.dumps(tool_output),
                "is_error" : False
            }
            tool_result_blocks.append(tool_result_block)
        except Exception as e:
            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_request.id,
                "content" : f"Error: {e}",
                "is_error" : True
            }
            
    return tool_result_blocks

In [39]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            batch_tool_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
        
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages
        

In [40]:
messages = []

add_user_message(
    messages,
    "set remainder for my doctors appointment , its 177 days after Jan 1st , 2050"
)

run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1st, 2050.
Now I'll set the reminder for your doctor's appointment on June 27, 2050:
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. You'll receive a notification at that time.


[{'role': 'user',
  'content': 'set remainder for my doctors appointment , its 177 days after Jan 1st , 2050'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1st, 2050.", type='text'),
   ToolUseBlock(id='toolu_013LNe9hQSERHyviZgJKQUoT', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013LNe9hQSERHyviZgJKQUoT',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set the reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_017kjZheip79PSX4zRtKQXbL', caller=DirectCaller(type='direct'), input={'content': 

In [41]:
messages = []

add_user_message(
    messages,
    """ set two remainder for March 1st , 2025  at 8AM:
        * I have a doctors appointment
        * I have taxes due
    """
)

run_conversation(messages)


Perfect! I've successfully set two reminders for March 1st, 2025 at 8:00 AM:

1. **I have a doctors appointment**
2. **I have taxes due**

Both reminders are now scheduled and you'll receive notifications at the specified time.


[{'role': 'user',
  'content': ' set two remainder for March 1st , 2025  at 8AM:\n        * I have a doctors appointment\n        * I have taxes due\n    '},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01MzSnPpGRvxRuWne3ToKKv6', caller=DirectCaller(type='direct'), input={'invocations': [{'name': 'set_reminder', 'arguments': '{"content": "I have a doctors appointment", "timestamp": "2025-03-01T08:00:00"}'}, {'name': 'set_reminder', 'arguments': '{"content": "I have taxes due", "timestamp": "2025-03-01T08:00:00"}'}]}, name='batch_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01MzSnPpGRvxRuWne3ToKKv6',
    'content': '[{"tool_name": "set_reminder", "output": null}, {"tool_name": "set_reminder", "output": null}]',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Perfect! I've successfully set two reminders for March 1st, 2025 at 8:00 AM:\n\n1. **I have a doctors appo

## Structured Data Tool Use

In [46]:
messages = [] 
add_user_message(messages, """ 
            Write a one-paragraph scholarly article about computer science. 
            Include a title and author name. 
        """)
response = chat(messages) 
text_from_message(response)

'# The Evolution of Machine Learning Architectures: From Statistical Methods to Deep Neural Networks\n\n**By Dr. Sarah Chen**\n\nThe field of machine learning has undergone a paradigmatic transformation over the past two decades, transitioning from classical statistical approaches to sophisticated deep neural network architectures that now dominate contemporary artificial intelligence research. Beginning with foundational techniques such as support vector machines and decision trees, the discipline has progressively embraced increasingly complex models capable of learning hierarchical representations of data, exemplified by convolutional neural networks and transformer-based architectures that have revolutionized computer vision and natural language processing respectively. This evolution has been catalyzed by three critical factors: exponential growth in computational resources, the availability of large-scale datasets, and algorithmic innovations including backpropagation refinements

In [49]:
messages = []

add_user_message(messages, text_from_message(response))
response = chat(messages, 
     tools=[article_summary_schema], 
     tool_choice={"type":"tool", "name" : "article_summary"},)

response.content[0].input

{'title': 'The Evolution of Machine Learning Architectures: From Statistical Methods to Deep Neural Networks',
 'author': 'Dr. Sarah Chen',
 'key_insights': ['Machine learning has evolved from classical statistical approaches (SVMs, decision trees) to sophisticated deep neural networks including CNNs and transformer architectures',
  'The transformation has been driven by three critical factors: exponential computational growth, large-scale datasets, and algorithmic innovations like backpropagation refinements and regularization techniques',
  'CNNs revolutionized computer vision while transformers transformed natural language processing',
  'Contemporary research focuses on addressing model interpretability, sample efficiency, and robustness to adversarial inputs',
  'Emerging subdisciplines like federated learning and neuromorphic computing extend ML capabilities to resource-constrained and edge computing environments',
  'As ML systems become integrated into critical applications (h